# Using EarthSciLab for calculations with the InMAP Source Receptor Matrix (ISRM)

## Introduction

In the past, there have been three ways to interact with ISRM: 1) downloading the ISRM file and working with it locally; 2) using the service hosted at inmap.run, and 3) using the [Zarr](https://zarr.dev/) version of ISRM hosted online.

You will always be able to use ISRM on your own computer, but we will be removing (free) access to the two online options in favor of a new service which we will describe in this post. There are two reasons for this: 1) as usage has increased, making the online services available for free has become a financial burden; and 2) the new service offers capabilities that the two old options do not.

### earthscilab.com

[EarthSciLab](https://earthscilab.com) is an online service that allows users to specify a model as a system of equations, and then run the model either in their browser or using cloud computing resources. It is built on top of [EarthSciAST](https://github.com/EarthSciML/EarthSciAST), an open-source system for compiling equations into runnable model simulations.

Simulations that are run in EarthSciLab using your browser are free, but the service charges for simulations run using cloud computing resources, so that it is able to recoup the cost of those resources. Because the ISRM is a large file stored in the cloud, doing the calculations in this notebook using EarthSciLab will cost you a little bit. However, for calculations of the scale included in this notebook, the cost is minimal.

### What this notebook is

One function:

```python
receptors = run_isrm(gdf)
```

`gdf` is a [GeoPandas](https://geopandas.org) GeoDataFrame of emissions — points, polygons or lines, in any CRS, with a column per pollutant in kg/yr. It comes back as a GeoDataFrame of the ISRM's 52,411 receptor cells carrying PM2.5 concentration and two mortality estimates.

Everything between those two frames — choosing the right `.esm` document, pruning the pathways your data does not feed, splitting emissions across multi-part geometries, cutting roads into segments, writing and uploading a shapefile, pricing the run, dispatching it and reading the answer back — is what the function does. Sections 1 and 2 build it; sections 3, 4 and 5 use it on real point, area and line inventories.

We will not set up the equations that turn emissions into concentrations and deaths. Those are pre-written `.esm` documents in [this repository](https://github.com/EarthSciML/isrm.esm), matching the calculations in the InMAP source code.

---
## 1. Plumbing

Standard library, plus `geopandas` (which brings `numpy`, `pandas`, `shapely` and
`pyproj`) and `matplotlib` to draw. The `.esm` documents are fetched from
[the repo](https://github.com/EarthSciML/isrm.esm) and everything the notebook
builds is held in memory, so it needs no checkout and writes nothing to disk —
except one temporary directory, because GDAL writes a shapefile as four files
with a shared stem.

In [ ]:
import base64
import copy
import io
import json
import pathlib
import ssl
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
import webbrowser
import zipfile

import earthsci_ast
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Polygon, box

API = "https://api.earthscilab.com"
WORKOS = "https://api.workos.com"
REPO_RAW = "https://raw.githubusercontent.com/EarthSciML/isrm.esm/main/"

_downloads = {}


def download(url):
    """`url`'s bytes, fetched once per kernel and kept in memory."""
    if url not in _downloads:
        with urllib.request.urlopen(url, timeout=900) as resp:
            _downloads[url] = resp.read()
        print(f"fetched {url.rsplit('/', 1)[-1]} ({len(_downloads[url]):,} bytes)")
    return _downloads[url]

### 1.1 HTTP

Two error types, because they are answered differently: an `HttpError` from
EarthSciLab is something to report, while a 400 from WorkOS carrying an OAuth
`error` code is part of the device-grant protocol — `authorization_pending` is
the normal case, not a failure.

In [ ]:
class HttpError(Exception):
    def __init__(self, status, body, url):
        super().__init__(f"HTTP {status} from {url}: {body[:600]}")
        self.status, self.body = status, body


class OAuthError(Exception):
    """A 400 from WorkOS carrying an OAuth 2.0 `error` code."""
    def __init__(self, error, description):
        super().__init__(f"{error}: {description}")
        self.error = error


def _open(req, timeout):
    try:
        return urllib.request.urlopen(req, timeout=timeout, context=ssl.create_default_context())
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")
        try:
            payload = json.loads(body)
        except ValueError:
            payload = {}
        if e.code == 400 and "error" in payload:
            raise OAuthError(payload["error"], payload.get("error_description", "")) from None
        raise HttpError(e.code, body, req.full_url) from None


def http_json(method, url, *, json_body=None, form=None, raw=None, headers=None, timeout=120.0):
    """One request, JSON in and JSON out (or `None` for an empty 204 body)."""
    data, hdrs = None, dict(headers or {})
    if json_body is not None:
        data = json.dumps(json_body).encode()
        hdrs["Content-Type"] = "application/json"
    elif form is not None:
        data = urllib.parse.urlencode(form).encode()
        hdrs["Content-Type"] = "application/x-www-form-urlencoded"
    elif raw is not None:
        data = raw
        hdrs["Content-Type"] = "application/octet-stream"
    req = urllib.request.Request(url, data=data, headers=hdrs, method=method)
    with _open(req, timeout) as resp:
        body = resp.read()
    return json.loads(body) if body else None

### 1.2 Signing in

**The OAuth 2.0 device authorization grant** — WorkOS's "CLI Auth", which AuthKit
serves with no extra configuration. It is the right flow here for a specific
reason rather than a stylistic one: a full-scale run takes about an hour and a
WorkOS *access* token is short-lived, so a hand-pasted token expires long before
the answer exists. The device grant hands back a **refresh** token, which this
class keeps for the life of the kernel, rotates on every use, and spends to mint a
fresh access token before each request — so a run that outlives its access token
still finishes. What it produces is an ordinary AuthKit user JWT — same JWKS, same
`sub` — so the API needed no change to accept it and runs bill to your own
account.

In [ ]:
class Session:
    """A signed-in EarthSciLab caller, refreshed on demand.

    The reason this is a class and not a header constant: "the token" is a thing
    that has to be re-derived, not held. Every request goes through `headers()`,
    which mints a new one whenever the current one is within a minute of expiry.
    """

    DEVICE_GRANT = "urn:ietf:params:oauth:grant-type:device_code"

    def __init__(self):
        self.client_id = http_json("GET", f"{API}/auth/config")["client_id"]
        self._access = None
        self._refresh_token = None

    @staticmethod
    def _expiry(token):
        """`exp` out of a JWT, read WITHOUT verifying it.

        Reading a claim to decide when to refresh is not the same act as trusting
        one: the API verifies this token against the JWKS, and a lie here can only
        cost an unnecessary refresh.
        """
        if not token:
            return 0.0
        try:
            payload = token.split(".")[1]
            payload += "=" * (-len(payload) % 4)
            return float(json.loads(base64.urlsafe_b64decode(payload)).get("exp", 0))
        except Exception:
            return 0.0

    def _adopt(self, response):
        self._access = response["access_token"]
        # Refresh tokens ROTATE. Keeping the new one is not housekeeping — hold on
        # to the old one and the next refresh fails.
        if response.get("refresh_token"):
            self._refresh_token = response["refresh_token"]

    def _refresh(self):
        if not self._refresh_token:
            return False
        try:
            self._adopt(http_json("POST", f"{WORKOS}/user_management/authenticate", form={
                "grant_type": "refresh_token", "refresh_token": self._refresh_token,
                "client_id": self.client_id}))
            return True
        except OAuthError:
            self._refresh_token = None
            return False

    def login(self):
        start = http_json("POST", f"{WORKOS}/user_management/authorize/device",
                          form={"client_id": self.client_id})
        print(f"\n  Your code is:  {start['user_code']}")
        print(f"  Open: {start['verification_uri_complete']}\n")
        try:
            webbrowser.open(start["verification_uri_complete"])
        except Exception:
            pass
        interval = float(start.get("interval", 5))
        deadline = time.time() + float(start.get("expires_in", 300))
        while time.time() < deadline:
            time.sleep(interval)
            try:
                self._adopt(http_json("POST", f"{WORKOS}/user_management/authenticate", form={
                    "grant_type": self.DEVICE_GRANT, "device_code": start["device_code"],
                    "client_id": self.client_id}))
                return
            except OAuthError as e:
                if e.error == "authorization_pending":
                    continue
                if e.error == "slow_down":
                    interval += 1
                    continue
                raise RuntimeError(f"sign-in refused: {e}") from None
        raise RuntimeError("sign-in timed out; run this cell again")

    def headers(self):
        if time.time() > self._expiry(self._access) - 60:
            if not self._refresh():
                self.login()
        return {"Authorization": f"Bearer {self._access}"}

    def get(self, path, timeout=120.0):
        return http_json("GET", f"{API}{path}", headers=self.headers(), timeout=timeout)

    def post(self, path, body=None, timeout=300.0):
        return http_json("POST", f"{API}{path}", json_body=body,
                         headers=self.headers(), timeout=timeout)

    def put_bytes(self, path, payload, timeout=600.0):
        return http_json("PUT", f"{API}{path}", raw=payload,
                         headers=self.headers(), timeout=timeout)

    def stream(self, path, timeout):
        headers = dict(self.headers(), Accept="text/event-stream")
        req = urllib.request.Request(f"{API}{path}", headers=headers, method="GET")
        return _open(req, timeout)

In [ ]:
session = Session()
print("signed in as", session.get("/me")["email"])
print("credit:", session.get("/credits"))

---
## 2. `run_isrm`

### 2.1 What your frame has to say

| column | units | meaning |
|---|---|---|
| `PM25` | kg/yr | primary PM2.5 → the `PrimaryPM25` pathway |
| `NOx` | kg/yr | → `pNO3` |
| `NH3` | kg/yr | → `pNH4` |
| `SOx` | kg/yr | → `pSO4` |
| `VOC` | kg/yr | → `SOA` |
| `STKHGT` | m | stack height — POINT frames only |
| `STKDIAM` | m | stack exit diameter |
| `STKTEMP` | K | exit gas temperature |
| `STKVEL` | m/s | exit gas velocity |

Bring whichever emission columns you have; at least one. A pollutant you do not
name is **pruned from the document**, which is worth more than tidiness — the
gated fetch of the source-receptor slabs is most of what a run costs, and it is
per pathway, so a PM2.5-only frame that left all five declared would pay for five
slabs in order to multiply four of them by zero.

All four stack columns or none. With them, a point source's mass is split across
the ISRM's three emission layers by ASME plume rise; without them everything goes
into layer 0, which understates ground-level impact for a tall stack and is the
only thing a frame with no stack data can support.

In [ ]:
# Each emission column, and the SR pathway it feeds.
PATHWAYS = {"PM25": "PrimaryPM25", "VOC": "SOA", "NOx": "pNO3",
            "NH3": "pNH4", "SOx": "pSO4"}
STACK = ["STKHGT", "STKDIAM", "STKTEMP", "STKVEL"]

TOTALS = ["TotalPM25", "deathsK", "deathsL"]
CELL = ["rcv_W", "rcv_S", "rcv_E", "rcv_N"]      # each receptor's own rectangle
OBSERVEDS = TOTALS + CELL

# The InMAP grid's own projection, and a SPHERE rather than an ellipsoid. The
# documents project lon/lat with these same parameters, so a length or an area
# measured here is the one the document measures.
LCC = ("+proj=lcc +lat_1=33 +lat_2=45 +lat_0=40 +lon_0=-97 "
       "+a=6370997 +b=6370997 +units=m +no_defs")

# The readers declare a file's CRS and reproject nothing, and the documents do
# the projection themselves, so the file goes out geographic.
FILE_CRS = "EPSG:4269"

FAMILIES = {"Point": "point", "MultiPoint": "point",
            "Polygon": "polygon", "MultiPolygon": "polygon",
            "LineString": "line", "MultiLineString": "line"}

TEMPLATES = {("point", True): "isrm_gdf_point.esm",
             ("point", False): "isrm_gdf_point_flat.esm",
             ("polygon", False): "isrm_gdf_polygon.esm",
             ("line", False): "isrm_gdf_line.esm"}

# `GET /datasets/{id}/field` clamps to this and a query cannot raise it. The
# receptor axis is 52,411, so a whole field fits and comes back at stride 1.
MAX_VALUES = 262_144

### 2.2 Preparing the layer

Three transformations, and each one exists because the reader or the engine
requires it rather than because it is tidy.

**One row per PART, emissions split between the parts.** The shapefile reader
emits one row per part and *replicates* the record's attributes onto each, so a
county of mainland-plus-islands would carry its whole emission two or three
times. `isrm_gdf_polygon.esm` says so, and says whose problem it is: *"a layer
where it does needs its emission divided among its parts before the file is
written, which is the layer builder's job and not this document's."* This
function is that layer builder.

**Interior rings dropped.** A shapefile writes a hole as another part, so the
reader would read it as its own record and *add* its emission share instead of
subtracting its area. Dropping the interiors keeps the mass exactly right and
overstates the footprint by the hole, which is the lesser error — and it is
reported rather than silent.

**Polylines cut into two-vertex segments.** `isrm_gdf_line.esm` bins per record,
and the projection-pushdown rewrite that makes this model affordable requires the
binning aggregate to declare exactly two ranges. A three-range aggregate over
(cell, road, segment) is not recognised, no support set is derived, and the whole
33 GB slab is fetched. So the segment has to be the record.

In [ ]:
def _explode(gdf, columns):
    """One row per part, with every emission column split among the parts."""
    multi = int(gdf.geom_type.str.startswith("Multi").sum())
    gdf = gdf.reset_index(drop=True)          # so the parent index is unique
    parts = gdf.explode(index_parts=False)    # ... and repeated once per part
    if len(parts) == len(gdf):
        return parts.reset_index(drop=True)

    measure = parts.geometry.area
    if not (measure > 0).any():
        measure = parts.geometry.length
    if not (measure > 0).any():               # points: all a point has is its count
        measure = pd.Series(1.0, index=parts.index)
    share = measure / measure.groupby(level=0).transform("sum")

    parts = parts.copy()
    for column in columns:
        parts[column] = parts[column].astype(float) * share
    print(f"  split {multi} multi-part record(s) into {len(parts) - len(gdf)} extra "
          f"part(s), emissions shared by area/length")
    return parts.reset_index(drop=True)


def _drop_holes(gdf):
    """Replace each polygon by its exterior ring."""
    holes = int(sum(len(g.interiors) for g in gdf.geometry))
    if not holes:
        return gdf
    gdf = gdf.copy()
    gdf["geometry"] = [Polygon(g.exterior) for g in gdf.geometry]
    print(f"  dropped {holes} interior ring(s); mass unchanged, footprint now "
          f"includes the holes")
    return gdf


def _segmentize(gdf, columns):
    """Cut every polyline into two-vertex segments, emission split by length."""
    rows = []
    for _, record in gdf.iterrows():
        coords = list(record.geometry.coords)
        segments = [LineString([a, b]) for a, b in zip(coords, coords[1:])]
        lengths = np.array([s.length for s in segments], dtype=float)
        share = (lengths / lengths.sum() if lengths.sum() > 0
                 else np.full(len(segments), 1.0 / len(segments)))
        for segment, fraction in zip(segments, share):
            rows.append({**{c: float(record[c]) * fraction for c in columns},
                         "geometry": segment})
    print(f"  cut {len(gdf)} polyline(s) into {len(rows)} two-vertex segment(s)")
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)


def normalize(gdf):
    """A frame the reader can be handed, and what the document needs to know."""
    if gdf.crs is None:
        raise ValueError("the frame has no CRS; set one so it can be reprojected to "
                         "the geographic CRS the documents project from")

    columns = [c for c in PATHWAYS if c in gdf.columns]
    if not columns:
        raise ValueError(f"no emission column: expected one or more of "
                         f"{list(PATHWAYS)} in kg/yr, got {list(gdf.columns)}")

    families = {FAMILIES.get(t) for t in gdf.geom_type.unique()}
    if len(families) != 1 or None in families:
        raise ValueError(f"one geometry family per frame; got "
                         f"{sorted(gdf.geom_type.unique())}")
    family = families.pop()

    stack = [c for c in STACK if c in gdf.columns]
    plume = family == "point" and len(stack) == len(STACK)
    if family == "point" and stack and not plume:
        raise ValueError(f"plume rise needs all of {STACK}; the frame has {stack}")

    # Work in the grid's own metres: every share below is a ratio of areas or
    # lengths, and the document computes them in this same projection.
    frame = _explode(gdf.to_crs(LCC), columns)
    if family == "polygon":
        frame = _drop_holes(frame)
    if family == "line":
        frame = _segmentize(frame, columns)

    keep = columns + (STACK if plume else [])
    frame = frame[keep + ["geometry"]].copy()
    for column in keep:
        frame[column] = np.asarray(frame[column], dtype=float)

    # The vertex axis the reader pads to, and the document declares.
    nvert_max = (1 if family == "point" else
                 2 if family == "line" else
                 max(len(g.exterior.coords) for g in frame.geometry))

    frame = frame.to_crs(FILE_CRS)
    print(f"  {len(frame):,} records · {family} · nvert_max {nvert_max} · "
          f"{', '.join(keep)}" + (" · ASME plume rise" if plume else ""))
    return frame, family, columns, plume, nvert_max


def shapefile_zip(frame):
    """The frame as a zipped four-file shapefile set, in memory.

    GDAL writes a shapefile as several files with a shared stem, so this is the
    one step that needs a directory — a temporary one, gone on the way out.
    `emis.shp` is the member every generated document names.
    """
    with tempfile.TemporaryDirectory() as tmp:
        frame.to_file(pathlib.Path(tmp) / "emis.shp", driver="ESRI Shapefile",
                      engine="pyogrio")
        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for part in sorted(pathlib.Path(tmp).iterdir()):
                archive.writestr(part.name, part.read_bytes())
    return buffer.getvalue()

### 2.3 The document

`isrm_gdf_point.esm` and its three siblings are generated from the repository's
published documents by
[`data/make_gdf_documents.py`](https://github.com/EarthSciML/isrm.esm/blob/main/data/make_gdf_documents.py) —
same physics, reading one uploaded shapefile with the column convention above
instead of an inventory in the inventory's own units. Three edits turn a template
into the document for *your* frame.

**The template library is folded in.** Each document reaches its shared body by
`{"ref": "./isrm_base.esm"}`, and a document that arrives in a request body has
no directory to be relative to — the server resolves `./` against *its* working
directory and rejects the document with `template-library file not found or
unreadable: /app/isrm_base.esm`. Merging the library's `expression_templates`
into the importing model and dropping the import key is the whole fix.

This is the one place the notebook does not hand the job to `earthsci_ast`.
`emit_document` is the spec's §9.6.4 emit and would be the obvious call, but it
also **closes the metaparameters** into the index sets — `emis_records` goes from
`{size: "N_REC"}` to `{size: 0}`. `N_REC` is discovered by the loader from the
file it is about to read, so a document arriving with that axis closed at its
declared default dies inside the engine on a zero-length axis.

**The loader is re-pointed**, at the dataset the upload just made, with the
`nvert_max` your geometry actually needs — in *both* the places that have to
agree, which is the subject of the comment in the cell below.

**Unused pathways are pruned**, as a reachability walk from the observeds the
document's own report block names rather than a list of names to delete —
dropping a pathway strands its `SR_`, `E_`, `conc_` and `pm_` variables and
nothing else, and a data source that no surviving variable reads is dropped too,
so no run discovers the extent of a file it will never look at. What a variable
*needs* comes from `earthsci_ast.free_variables`, which is not a convenience:
it sees through `apply_expression_template`, and a walk that stops at the call
site quietly prunes the projection constants.

In [ ]:
def equation_refs(doc):
    """`lhs` -> every variable its right-hand side needs.

    `earthsci_ast.free_variables` rather than a walk of our own, for one specific
    reason: it sees THROUGH `apply_expression_template`. `lcc_forward_x` declares
    `params: ["lon", "lat"]` and its body then reads `lat_1`, `lon_0`, `lcc_R`,
    `lcc_d2r` and `lcc_qp` straight out of the model scope; `krewski_deaths`
    reads `pop_scale` and `mort_scale` the same way. A walk that stops at the call
    site sees the bindings and none of those, concludes the projection constants
    are unreachable, and prunes them — which passes the schema AND `POST /quote`,
    then projects every source to garbage and dies in the fetch with no message
    beyond "the run ended without reporting a reason".
    """
    loaded = earthsci_ast.load_document(copy.deepcopy(doc))
    model = loaded.models[next(iter(doc["models"]))]
    return {eq.lhs: earthsci_ast.free_variables(eq.rhs) for eq in model.equations}


def prune_pathways(doc, columns):
    """Keep only the pathways the frame has columns for."""
    model = doc["models"]["ISRM"]
    variables, equations = model["variables"], model["equations"]
    report = doc["metadata"]["x_esd"]["report"]

    keep = [PATHWAYS[c] for c in columns]
    report["pathways"] = [p for p in report["pathways"] if p["sr_array"] in keep]

    total = next(e for e in equations if e["lhs"] == "TotalPM25")
    terms = [{"op": "index", "args": [f"conc_{p}", "rcv"]} for p in keep]
    total["rhs"]["args"] = [f"conc_{p}" for p in keep]
    total["rhs"]["expr"]["args"][1] = ({"op": "+", "args": terms} if len(terms) > 1
                                       else terms[0])

    roots = [report["total_pm25"], *report["deaths"].values(), report["record_field"],
             *CELL, "rcv_cx", "rcv_cy"]
    for pathway in report["pathways"]:
        roots += [pathway["concentration"], *pathway["emissions"],
                  "pm_" + pathway["sr_array"]]

    refs = equation_refs(doc)
    needed, stack = set(), list(roots)
    while stack:
        name = stack.pop()
        if name in needed or name not in variables:
            continue
        needed.add(name)
        stack.extend(refs.get(name, set()) & set(variables))

    dropped = sorted(set(variables) - needed)
    model["variables"] = {n: v for n, v in variables.items() if n in needed}
    model["equations"] = [e for e in equations if e["lhs"] in needed]
    live = {(v.get("update") or {}).get("source") for v in model["variables"].values()}
    for source in [s for s in doc["data_sources"] if s not in live]:
        doc["data_sources"].pop(source)
        dropped.append(source)
    if dropped:
        print(f"  pruned {len(dropped)} variable(s)/source(s); {len(keep)} pathway(s) "
              f"kept: {', '.join(keep)}")
    return doc


def fold_in_template_library(doc):
    """Merge the imported `expression_templates` in, and drop the import key.

    Deliberately by hand, and deliberately NOT `earthsci_ast.emit_document`,
    which is the esm-spec §9.6.4 emit and does more than this caller can
    tolerate: it CLOSES the metaparameters into the index sets, turning
    `emis_records: {size: "N_REC"}` into `{size: 0}`. `N_REC` is discovered by
    the loader from the file it is about to read, so a document that arrives with
    that axis closed at its declared default dies inside the engine on a
    zero-length axis. Touching the template table and nothing else is the whole
    transform that is wanted here.
    """
    for model in doc["models"].values():
        merged = {}
        for imported in model.pop("expression_template_imports", []) or []:
            library = json.loads(download(REPO_RAW + imported["ref"].rsplit("/", 1)[-1]))
            merged.update(library.get("expression_templates") or {})
        merged.update(model.get("expression_templates") or {})   # the document's own win
        model["expression_templates"] = merged
    return doc


def build_document(family, plume, columns, nvert_max, url, records=None):
    """The template, made into the document this particular frame needs."""
    name = TEMPLATES[(family, plume)]
    doc = fold_in_template_library(json.loads(download(REPO_RAW + name)))

    source = doc["data_sources"]["Emis"]
    source["source"]["url_template"] = url

    # `nvert_max` is TWO declarations that have to agree: what the reader pads the
    # vertex axis to, and how wide the document says that axis is. Setting only
    # the reader's half is not a validation error and not a wrong number — the
    # engine reads a [records, 5, 2] array into a [records, 56, 2] parameter and
    # the worker dies with no message at all. The axis is found through the
    # geometry variable rather than by name, because each family calls it
    # something different (N_EVERT, N_PVERT, N_LVERT).
    source["reader_options"]["nvert_max"] = nvert_max
    model = doc["models"]["ISRM"]
    geometry = next(name for name, v in model["variables"].items()
                    if ((v.get("update") or {}).get("from") or {})
                    .get("file_variable") == "geometry")
    axis = model["variables"][geometry]["shape"][1]
    doc["metaparameters"][doc["index_sets"][axis]["size"]]["default"] = nvert_max

    if records:
        # Scale is a DOCUMENT edit, not a request parameter: a loader-level
        # `select` range on the source that discovers its own extent.
        source["select"] = {"axes": [{"range": {"start": 0, "stop": records}}]}

    prune_pathways(doc, columns)

    # Cheaper to hear it from the library than from the API, which rejects an
    # invalid document with a bare "not valid under any of the schemas listed in
    # the 'oneOf' keyword" and 60 KB of echoed model, naming nothing.
    result = earthsci_ast.validate(earthsci_ast.load_document(copy.deepcopy(doc)))
    problems = list(result.schema_errors) + list(result.structural_errors)
    if problems:
        raise RuntimeError("the document this frame produced is invalid:\n  "
                           + "\n  ".join(str(p) for p in problems[:6]))

    print(f"  {name} · {axis} width {nvert_max} · valid"
          + (f" · first {records:,} records" if records else ""))
    return doc

### 2.4 Upload, price, run, read

`kind: "evaluate"`, not the default `"simulate"`. None of these documents has a
`D(·)` anywhere: `system_kind` is `nonlinear`, the analyses' time spans are
`0 -> 0`, and the whole answer is the observed graph. Dispatched as a simulation
the engine refuses the document outright with `Invalid parameter 'src_E'`, which
names the wrong thing entirely.

`POST /quote` needs no auth and no database; it is the pre-login preview. The
progress fraction is **phase**-weighted across eight `PreparePhase`s that differ
by four orders of magnitude in cost — `Rewrite` is milliseconds and `GatedFetch`
is tens of gigabytes off S3, so 88% is the fetch barely started. A long crawl
near the end is the I/O, not a hang.

In [ ]:
TERMINAL = {"succeeded", "failed", "cancelled", "capped"}


def money(dollars):
    return "—" if dollars is None else (
        f"${dollars:.4f}" if 0 < abs(dollars) < 0.01 else f"${dollars:.2f}")


def clock(seconds):
    seconds = int(seconds)
    if seconds >= 3600:
        return f"{seconds // 3600}h{seconds % 3600 // 60:02d}m"
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def upload_dataset(session, payload, name, esio_format="shapefile"):
    """Put bytes in the dataset store under `name`; return the committed record.

    Three calls, and the middle one is the bytes. `POST /datasets` mints the row
    and says where to write; the server derives everything it can rather than
    trusting the client — `format` and `origin` are the two exceptions, because
    deriving them would mean opening bytes this API has decided not to open.
    """
    created = session.post("/datasets", {"format": esio_format, "origin": "upload"})
    if created["upload"]["mode"] != "proxy":
        raise RuntimeError(f"this deployment wants a {created['upload']['mode']!r} "
                           "upload, not a proxied one")
    if len(payload) > created["max_object_bytes"]:
        raise RuntimeError(f"{name} is {len(payload):,} B, over the "
                           f"{created['max_object_bytes']:,} B per-object ceiling")
    session.put_bytes(f"{created['upload']['url']}?key={urllib.parse.quote(name)}",
                      payload)
    dataset = session.post(f"/datasets/{created['id']}/commit")
    print(f"  uploaded {name} ({len(payload):,} B) as dataset {dataset['id']}")
    return dataset


def dataset_url(dataset):
    """Where a document's `url_template` should point at this dataset."""
    base = dataset["store_url"].rstrip("/")
    return base if dataset.get("format") == "zarr" else f"{base}/{dataset['object_key']}"


def quote(doc, observeds):
    """Price the run. No auth — `POST /quote` has no database."""
    routing = http_json("POST", f"{API}/quote", timeout=300.0,
                        json_body={"esm": doc, "kind": "evaluate",
                                   "observeds": observeds})
    option = routing.get("dispatchable")
    if not option:
        raise RuntimeError(f"no dispatchable backend: {routing.get('reason')}")
    estimate, sizing = option["estimate"], routing.get("sizing") or {}
    print(f"  {option['backend']} · {sizing.get('vcpus')} vCPU / "
          f"{sizing.get('memory_mb')} MB · {clock(estimate['resource_seconds'])} "
          f"predicted (cap {clock(estimate['max_resource_seconds'])}) · "
          f"{money(estimate['price'])}")
    return estimate["price"]


def watch(session, run_id):
    """Follow a run to a terminal event, surviving a dropped connection.

    The stream replays everything already recorded before it streams, so a
    reconnect sees what it missed — including a terminal event that landed while
    we were disconnected. That is what makes reconnecting sufficient.
    """
    started = time.time()
    while True:
        try:
            with session.stream(f"/runs/{run_id}/events", timeout=240.0) as resp:
                for line in resp:
                    line = line.decode("utf-8", "replace").strip()
                    if not line.startswith("data:"):
                        continue
                    event = json.loads(line[5:]).get("kind") or {}
                    kind = event.get("type")
                    if kind == "progress":
                        f = event.get("fraction", 0.0)
                        print(f"\r  [{'#' * int(f * 40):<40}] {f * 100:5.1f}%  "
                              f"elapsed {clock(time.time() - started)}", end="", flush=True)
                    elif kind in ("queued", "started"):
                        print(f"  {kind}", flush=True)
                    elif kind in TERMINAL:
                        print()
                        return event
        except (HttpError, OAuthError, urllib.error.URLError, OSError, ValueError) as e:
            print(f"\n  (stream dropped: {e}; the run is server-side and unaffected)")
        run = session.get(f"/runs/{run_id}")
        if run["status"] in TERMINAL:
            return {"type": run["status"]}
        time.sleep(5)


def read_fields(session, dataset_id, names):
    """One 1-D array per name, out of the run's own Zarr store.

    An evaluate run writes `[eval(1), rcv_cells(52411)]`, so pinning every
    length-1 axis leaves exactly the receptor axis free — two free axes is a
    field, one is a line, and a line is what a column of the answer is.
    """
    dataset = session.get(f"/datasets/{dataset_id}")
    pins = ",".join(f"{d['name']}:0" for d in dataset.get("dims", []) if d["size"] == 1)
    series = {}
    for name in names:
        query = {"var": name, "max_values": MAX_VALUES}
        if pins:
            query["at"] = pins
        field = session.get(f"/datasets/{dataset_id}/field?{urllib.parse.urlencode(query)}")
        if len(field["axes"]) != 1 or field["axes"][0]["stride"] != 1:
            raise RuntimeError(f"{name}: expected one free axis read whole: "
                               f"{field['axes']}")
        series[name] = field["values"]
    return series


def receptor_frame(series):
    """The answer as a GeoDataFrame: one cell rectangle per receptor.

    Handed back in the grid's own Lambert conformal rather than in lon/lat,
    because the ISRM grid is variable-resolution — coarse over rural country,
    fine over cities — and its cells are rectangles in THESE metres. The mesh
    itself carries information, and reprojecting it to geographic would bend it.
    Call `.to_crs(...)` if you want it somewhere else.
    """
    cells = [box(w, s, e, n) for w, s, e, n in
             zip(series["rcv_W"], series["rcv_S"], series["rcv_E"], series["rcv_N"])]
    return gpd.GeoDataFrame({name: series[name] for name in TOTALS},
                            geometry=cells, crs=LCC)

### 2.5 The function

`records=N` truncates the layer to its first `N` records — the knob to turn when
you want minutes instead of the better part of an hour. `max_price` caps the bill;
left alone it is the quote, so a run cannot cost more than it was priced at.

In [ ]:
def run_isrm(gdf, records=None, max_price=None):
    """Run the InMAP ISRM over `gdf` on EarthSciLab, and bring the answer back.

    `gdf` carries one row per source, geometry in any CRS, and a column per
    pollutant in kg/yr (see 2.1). Returns a GeoDataFrame of the 52,411 receptor
    cells with TotalPM25 in µg/m³ and the two mortality estimates, and the run's
    id in `.attrs`.
    """
    print("layer")
    frame, family, columns, plume, nvert_max = normalize(gdf)

    print("upload")
    dataset = upload_dataset(session, shapefile_zip(frame), "emis.zip")

    print("document")
    doc = build_document(family, plume, columns, nvert_max,
                         dataset_url(dataset), records=records)

    print("quote")
    price = quote(doc, OBSERVEDS)

    print("run")
    run = session.post("/runs", {
        "esm": doc, "kind": "evaluate", "observeds": OBSERVEDS,
        "max_price": price if max_price is None else max_price})
    print(f"  run {run['id']} — {run['status']} on {run['backend']}, "
          f"{money(run['price'])}")
    outcome = watch(session, run["id"])
    if outcome["type"] != "succeeded":
        raise RuntimeError(f"run {run['id']} {outcome['type']}: "
                           f"{outcome.get('message', '')}")
    print(f"  succeeded in {clock(outcome.get('resource_seconds', 0))} of resource time")

    receptors = receptor_frame(read_fields(session, outcome["dataset_id"], OBSERVEDS))
    receptors.attrs.update(run_id=run["id"], dataset_id=dataset["id"],
                           records=len(frame), pathways=columns)
    for name in TOTALS:
        print(f"  sum({name})".ljust(22) + repr(float(receptors[name].sum())))
    return receptors

### 2.6 Drawing the answer

One rectangle per receptor cell, filled by concentration. Three choices, because
each could be made badly.

**One hue, light to dark.** Concentration is a *magnitude*, and magnitude gets a
sequential ramp — not a rainbow, which implies category boundaries the data does
not have and is unreadable to a colorblind viewer.

**The scale is clipped at a percentile, and says so.** PM2.5 over this grid is
extremely skewed: a handful of cells beside large sources sit orders of magnitude
above the median, so a scale stretched to the true maximum renders essentially the
whole country as the lightest step. The caption prints the real maximum.

**Cropped to the cells that carry the mass.** An inventory covering one state
still produces a field over the whole national grid. Cells are taken in
descending order until they account for `share` of the total rather than
thresholded at a fraction of the peak, because a source–receptor field's tail is
thin but very wide — almost every cell in the country sits above any small
fraction of the peak, so a threshold rule crops nothing exactly when cropping is
wanted most. Pass `share=None` to see the whole grid.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# `Blues` with its white end trimmed off: at pure white the lowest cells are
# invisible against the page.
SEQUENTIAL = ListedColormap(plt.get_cmap("Blues")(np.linspace(0.15, 1.0, 256)))


def show(receptors, title="", column="TotalPM25", units="µg/m³",
         clip_pct=99.0, share=0.90, height=6.4):
    """The receptor grid as a choropleth."""
    v = receptors[column].to_numpy()
    vmax = np.percentile(v, clip_pct) or v.max() or 1.0

    bounds = receptors.total_bounds
    if share:
        order = np.argsort(v)[::-1]
        keep = order[:np.searchsorted(np.cumsum(v[order]), v.sum() * share) + 1]
        bounds = receptors.iloc[keep].total_bounds
    x0, y0, x1, y1 = bounds
    pad = 0.06 * max(x1 - x0, y1 - y0)

    # The figure follows the MAP's shape: these are metres in both directions, so
    # the axes are equal-aspect and their width follows the extent.
    span = height * (x1 - x0) / (y1 - y0)
    fig, ax = plt.subplots(figsize=(min(13.0, max(4.5, span)) + 1.8, height), dpi=150)

    # No edge colour: at 52,411 cells a stroke per rectangle is most of the ink on
    # the page, and the boundaries it draws are the grid's, not the data's.
    receptors.plot(ax=ax, column=column, cmap=SEQUENTIAL, vmin=0.0, vmax=vmax,
                   linewidth=0.0, edgecolor="none", rasterized=True, legend=True,
                   legend_kwds={"label": f"{column} ({units})", "shrink": 0.55,
                                "extend": "max" if v.max() > vmax else "neither"})
    ax.set_xlim(x0 - pad, x1 + pad)
    ax.set_ylim(y0 - pad, y1 + pad)
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(
        f"{title or column}\n"
        f"{receptors.attrs.get('records', len(receptors)):,} emission records · "
        f"{', '.join(receptors.attrs.get('pathways', []))} · "
        f"run {receptors.attrs.get('run_id', '?')[:8]}\n"
        f"scale clipped at the {clip_pct:g}th percentile ({vmax:.3g} {units}); "
        f"true maximum {v.max():.3g} {units}",
        fontsize=9, loc="left")
    fig.tight_layout()
    return ax

---
## 3. Point sources — the EGU inventory

The EPA's 2016fd alpha point-source FF10 inventory, filtered to electricity
generating units. This is the shape of the real problem: an inventory in
somebody else's format, on somebody else's server, in units of its own.

FF10 is **long** — one row per (stack, pollutant) — and `run_isrm` wants **wide**,
one row per stack with a column per pollutant. Pivoting needs to know which of
FF10's 77 pollutant codes belongs to which pathway, and rather than write that
table out again, the cell below reads it out of `isrm_point.esm`: the published
document classifies a row by comparing its integer code against a list per
pathway (`is_VOC` and friends), and maps `POLID` strings onto those integers with
its loader variable's own `codes` map. Read both back and this pivot cannot
disagree with the document that defines the classification.

The zip is ~72 MB, so the first run of this cell takes a minute.

In [ ]:
FF10_URL = ("https://gaftp.epa.gov/air/emismod/2016/alpha/2016fd/emissions/"
            "2016fd_inputs_point.zip")

# FF10 point columns are positional (0-based), as EarthSciIO's own reader has
# them; isrm_point.esm's metadata.x_esd.columns names the same indices.
FF10 = {12: "POLID", 13: "ANN_VALUE", 17: "STKHGT", 18: "STKDIAM",
        19: "STKTEMP", 21: "STKVEL", 23: "LONGITUDE", 24: "LATITUDE"}

SHORT_TON_KG = 907.18474


def _compared_codes(node, found=None):
    """Every integer an `==` in this expression compares against.

    Collected by walking rather than by indexing, because the shape of a mask
    depends on how many codes it covers: `is_VOC` is an `or` of 35 `==` nodes
    and `is_NH3` is a single bare `==`.
    """
    found = set() if found is None else found
    if isinstance(node, dict):
        if node.get("op") == "==":
            args = node.get("args") or []
            if len(args) == 2 and isinstance(args[1], (int, float)):
                found.add(int(args[1]))
        for value in node.values():
            _compared_codes(value, found)
    elif isinstance(node, list):
        for item in node:
            _compared_codes(item, found)
    return found


def pollutant_classes():
    """`POLID` -> emission column, read out of isrm_point.esm itself."""
    model = json.loads(download(REPO_RAW + "isrm_point.esm"))["models"]["ISRM"]
    codes = model["variables"]["pollutant"]["update"]["from"]["codes"]["map"]
    classes = {}
    for column in PATHWAYS:
        mask = next(e for e in model["equations"] if e["lhs"] == f"is_{column}")
        wanted = _compared_codes(mask["rhs"])
        for polid, code in codes.items():
            if code in wanted:
                classes[polid.upper()] = column
    return classes


def egu_points(limit=None):
    """The EGU inventory as one row per stack, in SI units."""
    classes = pollutant_classes()
    with zipfile.ZipFile(io.BytesIO(download(FF10_URL))) as archive:
        member = next(n for n in archive.namelist()
                      if "egu" in n.lower() and not n.endswith("/"))
        raw = pd.read_csv(io.BytesIO(archive.read(member)), header=None, comment="#",
                          usecols=list(FF10), names=None, dtype=str,
                          engine="python", on_bad_lines="skip")
    raw = raw.rename(columns=FF10)
    # One asserted header line survives the '#' comments.
    if str(raw.iloc[0]["POLID"]).strip().lower() in ("polid", "poll"):
        raw = raw.iloc[1:]
    print(f"  {member}: {len(raw):,} FF10 rows")

    raw["column"] = raw["POLID"].str.strip().str.upper().map(classes)
    raw = raw.dropna(subset=["column"])
    numeric = ["ANN_VALUE", "STKHGT", "STKDIAM", "STKTEMP", "STKVEL",
               "LONGITUDE", "LATITUDE"]
    for name in numeric:
        raw[name] = pd.to_numeric(raw[name], errors="coerce")
    raw = raw.dropna(subset=numeric)
    print(f"  {len(raw):,} rows classified into {sorted(raw['column'].unique())}")

    # A stack IS its location plus its four parameters: rows that agree on all
    # six are the same physical stack, and summing them is exact for a linear
    # model. Pivoting on that key is what turns long into wide.
    key = ["LONGITUDE", "LATITUDE", "STKHGT", "STKDIAM", "STKTEMP", "STKVEL"]
    wide = (raw.pivot_table(index=key, columns="column", values="ANN_VALUE",
                            aggfunc="sum", fill_value=0.0)
               .reset_index())
    wide.columns.name = None
    print(f"  pivoted to {len(wide):,} stacks")
    if limit:
        wide = wide.head(limit)

    emissions = [c for c in PATHWAYS if c in wide.columns]
    for column in emissions:
        wide[column] = wide[column] * SHORT_TON_KG              # short ton/yr -> kg/yr
    wide["STKHGT"] = wide["STKHGT"] * 0.3048                    # ft -> m
    wide["STKDIAM"] = wide["STKDIAM"] * 0.3048
    wide["STKVEL"] = wide["STKVEL"] * 0.3048                    # ft/s -> m/s
    wide["STKTEMP"] = (wide["STKTEMP"] - 32.0) * 5.0 / 9.0 + 273.15   # degF -> K

    return gpd.GeoDataFrame(
        wide[emissions + STACK],
        geometry=gpd.points_from_xy(wide["LONGITUDE"], wide["LATITUDE"]),
        crs="EPSG:4269")

In [ ]:
STACKS = 200          # <- None for every stack in the inventory (the better part of an hour)

points = egu_points(limit=STACKS)
print()
print(points[[c for c in PATHWAYS if c in points.columns]].sum().to_string())
points.head(3)

In [ ]:
point_receptors = run_isrm(points)

In [ ]:
show(point_receptors, "PM2.5 from EGU point sources, through the InMAP ISRM")
plt.show()

---
## 4. Area sources — county polygons

The same function, a different geometry family. Census county boundaries carrying
a uniform per-square-kilometre emission rate — an **example** quantity: the point
of the area path is the geometry, how a polygon's mass reaches the grid cells it
covers, not the inventory.

One emission column, so `run_isrm` prunes four of the five pathways and the run
fetches one source-receptor slab instead of five.

In [ ]:
COUNTIES_URL = ("https://www2.census.gov/geo/tiger/GENZ2020/shp/"
                "cb_2020_us_county_20m.zip")
STATE = "17"           # Illinois
RATE_PM25 = 1000.0     # kg/yr per km² of land area — an example rate

counties = gpd.read_file(io.BytesIO(download(COUNTIES_URL)))
counties = counties[counties["STATEFP"] == STATE].copy()
counties["PM25"] = RATE_PM25 * counties["ALAND"] / 1e6
polygons = counties[["NAME", "PM25", "geometry"]]
print(f"{len(polygons)} counties · {polygons['PM25'].sum():,.0f} kg/yr of primary PM2.5")
polygons.head(3)

In [ ]:
polygon_receptors = run_isrm(polygons)

In [ ]:
show(polygon_receptors, "PM2.5 from an Illinois county area-source layer")
plt.show()

---
## 5. Line sources — interstates

TIGER primary roads, filtered to interstates, carrying emissions per kilometre of
road. `run_isrm` simplification is *your* job — a road at full TIGER resolution
has vertices every few metres, and the ISRM's finest cell is a kilometre across,
so the cell below thins the geometry to a fifth of that before handing it over.
Everything after that is the function's: the roads get cut into two-vertex
segments with each road's emission apportioned by length.

Two emission columns this time, so three pathways get pruned rather than four.

In [ ]:
ROADS_URL = (f"https://www2.census.gov/geo/tiger/TIGER2020/PRISECROADS/"
             f"tl_2020_{STATE}_prisecroads.zip")
SIMPLIFY_M = 200.0     # a fifth of the ISRM grid's finest cell
RATE_ROAD_PM25 = 40.0     # kg/yr per km of road — example rates, as above
RATE_ROAD_NOX = 400.0

roads = gpd.read_file(io.BytesIO(download(ROADS_URL)))
interstates = roads[(roads["MTFCC"] == "S1100") & (roads["RTTYP"] == "I")].to_crs(LCC)
interstates["geometry"] = interstates.simplify(SIMPLIFY_M)

km = interstates.length / 1000.0
interstates["PM25"] = RATE_ROAD_PM25 * km
interstates["NOx"] = RATE_ROAD_NOX * km
lines = interstates[["FULLNAME", "PM25", "NOx", "geometry"]]
print(f"{len(lines)} road(s) · {km.sum():,.0f} km · "
      f"{lines['PM25'].sum():,.0f} kg/yr PM2.5, {lines['NOx'].sum():,.0f} kg/yr NOx")
lines.head(3)

In [ ]:
line_receptors = run_isrm(lines)

In [ ]:
show(line_receptors, "PM2.5 from Illinois interstate line sources")
plt.show()

---
## 6. Cleaning up

Each run uploaded a layer as an unsaved dataset, which carries an `expires_at`
and is billed for storage until then. `POST /datasets/{id}/save` keeps one
indefinitely; `DELETE` removes it now.

In [ ]:
for receptors in (point_receptors, polygon_receptors, line_receptors):
    dataset_id = receptors.attrs["dataset_id"]
    http_json("DELETE", f"{API}/datasets/{dataset_id}", headers=session.headers())
    print("deleted", dataset_id)